# Notebook 3：结果分析与研究报告

**项目**：P2PNet-Soy 大豆种子定位与计数复现  
**论文**：Zhao et al., Plant Phenomics, 2022

本 Notebook 完成以下内容：
1. 计算核心评估指标（MAE、RMSE、R²）
2. 预测值与真实值对比分析
3. 误差分布可视化
4. 与原文结果对比
5. 结论与讨论

## 1. 环境准备

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = Path('../outputs/results')
FIGURES_DIR = Path('../outputs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('环境加载完成 ✓')

## 2. 加载预测结果

> 此处加载由 `02_model_inference.ipynb` 生成的预测结果文件

In [ ]:
# 加载推理结果
results_file = RESULTS_DIR / 'predictions.csv'

if results_file.exists():
    df = pd.read_csv(results_file)
    print(f'加载结果：{len(df)} 条预测记录')
    print(df.head())
else:
    print('⚠️  未找到预测结果文件，请先运行 02_model_inference.ipynb')
    print('或使用模拟数据演示流程：')
    
    # 用模拟数据演示分析流程
    np.random.seed(42)
    n = 100
    gt = np.random.randint(20, 200, n).astype(float)
    pred = gt + np.random.normal(0, 15, n)  # 模拟 MAE ≈ 12.94
    pred = np.clip(pred, 0, None)
    
    df = pd.DataFrame({
        'filename': [f'sample_{i}' for i in range(n)],
        'gt_count': gt,
        'pred_count': pred
    })
    print('（使用模拟数据演示）')

## 3. 核心评估指标计算

In [ ]:
gt   = df['gt_count'].values
pred = df['pred_count'].values

mae  = mean_absolute_error(gt, pred)
rmse = np.sqrt(mean_squared_error(gt, pred))
r2   = r2_score(gt, pred)
mre  = np.mean(np.abs(gt - pred) / (gt + 1e-6)) * 100  # 平均相对误差

print('=' * 45)
print(f'  评估指标              复现结果    原文结果')
print('=' * 45)
print(f'  MAE（均绝对误差）     {mae:>8.2f}    12.94')
print(f'  RMSE（均方根误差）    {rmse:>8.2f}    --')
print(f'  R²（决定系数）        {r2:>8.4f}    --')
print(f'  MRE（均相对误差）     {mre:>7.2f}%    --')
print('=' * 45)

## 4. 预测值 vs 真实值散点图

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- 左图：散点图 ---
ax = axes[0]
ax.scatter(gt, pred, alpha=0.6, color='steelblue', s=30, edgecolors='white', linewidths=0.5)

# 理想对角线
min_val, max_val = min(gt.min(), pred.min()), max(gt.max(), pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='理想预测（y=x）')

# 线性拟合
slope, intercept, r, p, se = stats.linregress(gt, pred)
x_line = np.linspace(min_val, max_val, 100)
ax.plot(x_line, slope * x_line + intercept, 'orange', linewidth=1.5,
        label=f'拟合线（R²={r2:.3f}）')

ax.set_xlabel('真实种子数量（Ground Truth）', fontsize=12)
ax.set_ylabel('预测种子数量（Prediction）', fontsize=12)
ax.set_title('预测值 vs 真实值', fontsize=13)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 右图：误差分布
ax = axes[1]
errors = pred - gt
ax.hist(errors, bins=30, color='coral', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
ax.axvline(errors.mean(), color='steelblue', linestyle='-', linewidth=1.5,
           label=f'均值 = {errors.mean():.2f}')
ax.set_xlabel('预测误差（Prediction − GT）', fontsize=12)
ax.set_ylabel('样本数量', fontsize=12)
ax.set_title('预测误差分布', fontsize=13)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'prediction_analysis.png', bbox_inches='tight')
plt.show()
print('图表已保存至 outputs/figures/prediction_analysis.png')

## 5. 与原文结果对比

In [ ]:
# 原文报告的方法对比数据（Table 来自论文）
methods = {
    '原始 P2PNet': 105.55,
    'P2PNet-Soy\n（论文结果）': 12.94,
    'P2PNet-Soy\n（复现结果）': mae,
}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(methods.keys(), methods.values(),
              color=['#aec6cf', '#4a90d9', '#e8734a'],
              edgecolor='white', width=0.5)

for bar, val in zip(bars, methods.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('MAE（均绝对误差）', fontsize=12)
ax.set_title('各方法 MAE 对比（数值越低越好）', fontsize=13)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, max(methods.values()) * 1.2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mae_comparison.png', bbox_inches='tight')
plt.show()

## 6. 结论与讨论

### 6.1 复现结果总结

本项目对 P2PNet-Soy（Zhao et al., 2022）进行了复现，主要结论如下：

| 方面 | 结论 |
|------|------|
| **数据可重复性** | 数据集通过官方渠道申请可获取，格式清晰（PNG图像 + TXT点标注） |
| **代码可重复性** | 原始代码基于 Jupyter Notebook 发布，依赖项明确 |
| **环境可重复性** | 需要 PyTorch 环境，建议使用 Conda 管理依赖 |
| **结果可重复性** | MAE 复现结果为 **[待填写]**，与论文报告的 12.94 相比误差为 **[待填写]** |

### 6.2 方法创新点理解

P2PNet-Soy 相比原始 P2PNet 的核心改进在于：

1. **问题特殊性**：大豆种子在田间拍摄图像中存在遮挡、密集分布等问题，传统 P2PNet 无法有效处理
2. **多尺度特征**：引入空洞卷积与低层特征，使模型对不同大小的种子更鲁棒
3. **后处理优化**：无监督聚类减少了过密预测点的重复计数问题

### 6.3 局限性

- 数据集需要申请获取，无法完全自动化复现流程
- 原文未提供预训练模型权重，需要重新训练（训练耗时较长）
- 随机种子可能导致训练结果略有差异

### 6.4 参考文献

Zhao J, Kaga A, Hirafuji M, Ninomiya S, Guo W. Improved Field-Based Soybean Seed Counting and Localization with Feature Level Considered. *Plant Phenomics*. 2022. https://doi.org/10.34133/plantphenomics.0026